# Predicao do peso de animais por regressao

Objetivo: treinar um modelo de regressao para estimar o `PESO` do animal a partir das caracteristicas disponiveis na base.

## 1. Importacao das bibliotecas

In [111]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler

## 2. Carregamento dos dados

In [112]:
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT_DIR / 'data' / 'raw' / 'dados_ultrassom_animais.csv'

df = pd.read_csv(DATA_PATH)
df.head()

,ANIMAL,PAI,MÃE,P. RACIAL,SEXO,DN,IDADE,PROPRIETÁRIO,PROPRIEDADE,MUNICÍPIO,...,LR,LCAB,LIL,LIS,Cga,Cper,PerPe,Ccau,DC,CE
0,1903304027,1901102022,1.903303e+09,ANGLO,F,8/12/2004,3406,CAF/UFPI,CAF/UFPI,FLORIANO/PI,...,NaN,NaN,"18,0","14,0","20,0","33,0","40,0","13,0","11,0",NaN
1,1903307023,1901103032,1.903305e+09,ANGLO,F,3/20/2007,2456,CAF/UFPI,CAF/UFPI,FLORIANO/PI,...,NaN,NaN,"18,0","18,5","19,0","37,0","38,0","14,0","10,0",NaN
2,1903309056,1901103032,1.903304e+09,ANGLO,F,5/15/2009,1669,CAF/UFPI,CAF/UFPI,FLORIANO/PI,...,NaN,NaN,"18,0","20,0","20,0","35,0","41,0","14,0","9,0",NaN
3,1903309054,1901103032,1.903305e+09,ANGLO,F,5/13/2009,1671,CAF/UFPI,CAF/UFPI,FLORIANO/PI,...,NaN,NaN,"17,0","18,0","20,0","39,0","10,0","14,0","8,0",NaN
4,1903308003,1901703001,1.903306e+09,ANGLO,F,5/2/2008,2047,CAF/UFPI,CAF/UFPI,FLORIANO/PI,...,NaN,NaN,"15,0","14,0","17,0","41,0","34,0","19,0","8,0",NaN


In [113]:
df.shape

(214, 39)

In [114]:
(df.isna().mean() * 100).sort_values(ascending=False).head(20)

LCAB            98.130841
LR              98.130841
EGS (mm)        98.130841
CE              92.056075
DN              17.757009
MÃE             15.887850
PAI             14.953271
IDADE            0.000000
SEXO             0.000000
P. RACIAL        0.000000
ANIMAL           0.000000
EST. FISIO       0.000000
PROPRIETÁRIO     0.000000
MUNICÍPIO        0.000000
PROPRIEDADE      0.000000
POL (cm)         0.000000
RATIO (cm)       0.000000
EGE (mm)         0.000000
MOL              0.000000
EC               0.000000
dtype: float64

## 3. Pre-processamento

- O rotulo (`y`) sera a coluna `PESO`.
- Colunas sem nome e colunas com muitos dados faltantes serao removidas.
- Numeros com virgula decimal serao convertidos para formato numerico.
- Identificadores serao removidos para evitar que o modelo memorize registros.
- **Valores numéricos serão normalizados (escala Min-Max) para que variáveis com grandezas diferentes não desequilibrem o modelo.**

In [115]:
TARGET = 'PESO'
MISSING_THRESHOLD = 0.50
RANDOM_STATE = 50
SELECTED_FEATURES = [
    'AC', 'AG', 'CC', 'AP', 'P.C', 'CT', 'CO', 'CCAB',
    'LIL', 'LIS', 'Cga', 'Cper', 'PerPe', 'Ccau', 'DC',
]

def convert_decimal_columns(dataframe):
    dataframe = dataframe.copy()
    for column in dataframe.columns:
        if pd.api.types.is_object_dtype(dataframe[column]) or pd.api.types.is_string_dtype(dataframe[column]):
            values = dataframe[column].astype('string').str.strip()
            numeric_values = pd.to_numeric(
                values.str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
                errors='coerce',
            )
            original_not_null = values.notna().sum()
            converted_not_null = numeric_values.notna().sum()
            if original_not_null > 0 and converted_not_null / original_not_null >= 0.80:
                dataframe[column] = numeric_values
    return dataframe

def clean_data(dataframe):
    dataframe = dataframe.copy()
    dataframe = dataframe.drop(columns=[col for col in dataframe.columns if col.startswith('Unnamed')], errors='ignore')
    dataframe = convert_decimal_columns(dataframe)
    dataframe = dataframe.dropna(subset=[TARGET])
    missing_ratio = dataframe.isna().mean()
    dataframe = dataframe.drop(columns=missing_ratio[missing_ratio > MISSING_THRESHOLD].index, errors='ignore')
    dataframe = dataframe[[*SELECTED_FEATURES, TARGET]].copy()
    return dataframe

# def normalize_data(dataframe):
#     dataframe = dataframe.copy()
#     for column in dataframe.columns:
#         if pd.api.types.is_numeric_dtype(dataframe[column]):
#             min_val = dataframe[column].min()
#             max_val = dataframe[column].max()
#             if max_val > min_val:
#                 dataframe[column] = (dataframe[column] - min_val) / (max_val - min_val)
#     return dataframe

clean_df = clean_data(df)
clean_df.head()

,AC,AG,CC,AP,P.C,CT,CO,CCAB,LIL,LIS,Cga,Cper,PerPe,Ccau,DC,PESO
0,76.0,76.0,80.0,40.0,10.0,91.0,19.0,24.0,18.0,14.0,20.0,33.0,40.0,13.0,11.0,60.0
1,75.0,74.0,77.0,37.0,10.0,91.0,21.0,24.0,18.0,18.5,19.0,37.0,38.0,14.0,10.0,38.0
2,72.0,74.0,77.0,35.0,10.0,88.0,21.0,26.0,18.0,20.0,20.0,35.0,41.0,14.0,9.0,50.0
3,67.0,72.0,76.0,35.0,10.0,83.0,20.0,26.0,17.0,18.0,20.0,39.0,10.0,14.0,8.0,52.0
4,73.0,74.0,74.0,38.0,9.0,84.0,21.0,25.0,15.0,14.0,17.0,41.0,34.0,19.0,8.0,42.0


In [116]:
clean_df.shape

(214, 16)

## 4. Divisao treino/teste

A divisao sera de 70% para treino e 30% para teste.

In [117]:
X = clean_df.drop(columns=[TARGET])
y = clean_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
)

X_train.shape, X_test.shape

((149, 15), (65, 15))

## 5. Pipeline de modelagem

In [118]:
# Apenas a lista de numéricos será usada agora
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()

# 1. Variáveis categóricas comentadas, pois o banco é 100% numérico
# categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

# ==========================================
# PIPELINE DE NÚMEROS (Ativo com Normalização)
# ==========================================
numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler()), # Normalização Min-Max ativa
        # ('scaler', StandardScaler()), # Variante Z-Score (Padronização) comentada
    ]
)

# ==========================================
# PIPELINE DE CATEGORIAS/TEXTOS (Comentado)
# ==========================================
# categorical_pipeline = Pipeline(
#     steps=[
#         ('imputer', SimpleImputer(strategy='most_frequent')),
#         ('encoder', OneHotEncoder(handle_unknown='ignore')),
#     ]
# )

# Junta os pipelines no preprocessor final
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        # ('categorical', categorical_pipeline, categorical_features), # Comentado!
    ]
)

In [119]:
models = {
        "Regressao Linear (Exata)": LinearRegression(),
        "Regressao Linear (Gradient Descent)": SGDRegressor(
            max_iter=1000,      # Número máximo de passadas nos dados
            tol=1e-3,           # Critério de parada (quando o erro parar de cair)
            random_state=RANDOM_STATE
        ),
}

results = []
trained_models = {}

for name, estimator in models.items():
    model = Pipeline(steps=[('preprocessor', preprocessor), ('model', estimator)])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[name] = model
    results.append({
        'modelo': name,
        'mae': mean_absolute_error(y_test, predictions),
        'rmse': root_mean_squared_error(y_test, predictions),
        'r2': r2_score(y_test, predictions),
    })

metrics = pd.DataFrame(results).sort_values('rmse')
metrics

,modelo,mae,rmse,r2
1,Regressao Linear (Gradient Descent),4.571603,6.239007,0.733846
0,Regressao Linear (Exata),4.595457,6.370948,0.722469


## 6. Salvamento dos resultados

In [120]:
best_model_name = metrics.iloc[0]['modelo']
best_model = trained_models[best_model_name]

(ROOT_DIR / 'data' / 'processed').mkdir(parents=True, exist_ok=True)
(ROOT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(ROOT_DIR / 'reports').mkdir(parents=True, exist_ok=True)

clean_df.to_csv(ROOT_DIR / 'data' / 'processed' / 'dados_limpos.csv', index=False)
metrics.to_csv(ROOT_DIR / 'reports' / 'metricas_modelo.csv', index=False)
joblib.dump(best_model, ROOT_DIR / 'models' / 'modelo_peso.joblib')

best_model_name

'Regressao Linear (Gradient Descent)'